In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

ModuleNotFoundError: No module named 'numpy'

In [6]:
pip install pandas numpy scikit-learn implicit tqdm

  Using cached pandas-2.3.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached numpy-2.3.0-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached scikit_learn-1.7.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (31 kB)
  Using cached implicit-0.7.2.tar.gz (70 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.15.3-cp312-cp312-macosx_14_0_arm64.whl.metadata (61 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 9.8 MB/s eta 0:00:00a 0:00:01
Using cached numpy-2.3.0-cp312-cp312-macosx_14_0_arm64.whl (5.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 7.0 MB/s eta 0:00:0000:0

In [35]:
import pandas as pd

train_data = pd.read_csv('hse_train.csv')
train_data['rank'] = train_data.groupby('user_id')['timestamp'].rank(method='dense', ascending=True)
train_data['weight'] = 1 / train_data['rank']
train_data.head()

,user_id,item_id,timestamp,rank,weight
0,258671,74254,1511701649,1.0,1.000000
1,258671,115615,1511841435,2.0,0.500000
2,258671,176624,1512105022,3.0,0.333333
3,240498,45484,1511605442,1.0,1.000000
4,240498,39504,1511756830,2.0,0.500000


In [32]:
train_data['timestamp'].max()

Timestamp('1970-01-01 00:00:01.512259199')

In [33]:
train_data['timestamp'].min()

Timestamp('1970-01-01 00:00:01.511539200')

In [ ]:
# todo remove or upd later

# train_data = train_data.drop_duplicates(subset=['user_id', 'item_id'])

In [ ]:
# assert train_data.duplicated(subset=['user_id', 'item_id']).sum() == 0

In [5]:
train_data.shape

(4630328, 3)

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse import csr_matrix
import numpy as np

user_ids = train_data['user_id'].unique()
item_ids = train_data['item_id'].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}

item_to_idx_inv = {v: k for k, v in item_to_idx.items()}

train_data['user_idx'] = train_data['user_id'].map(user_to_idx)
train_data['item_idx'] = train_data['item_id'].map(item_to_idx)

assert train_data['user_idx'].min() == 0, "Индексы пользователей должны начинаться с нуля"
assert train_data['item_idx'].min() == 0, "Индексы товаров должны начинаться с нуля"

interaction_matrix = coo_matrix(
    (np.ones(len(train_data)), (train_data['user_idx'], train_data['item_idx'])),
    shape=(len(user_ids), len(item_ids))
)

In [14]:
user_ids.shape

(701981,)

In [15]:
item_ids.shape

(180599,)

In [16]:
interaction_matrix.shape

(701981, 180599)

In [24]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=20, random_state=42)
model.fit(interaction_matrix)

/Users/luka.markov/git/luka/RecSys_course/myenv/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.1206200122833252 seconds
  warnings.warn(
100%|██████████| 20/20 [00:50<00:00,  2.54s/it]


In [25]:
item_to_idx_inv = {v: k for k, v in item_to_idx.items()}

def recommend_for_user(user_idx, model, interaction_matrix, top_n=20):
    if user_idx >= interaction_matrix.shape[0]:
        print(f"User index {user_idx} is out of bounds. Skipping.")
        return []
    
    user_items = interaction_matrix[user_idx]
    try:
        item_ids, scores = model.recommend(user_idx, user_items, N=top_n)
    except IndexError as e:
        print(f"IndexError while recommending for user {user_idx}: {e}")
        return []
    
    valid_item_ids = [item_idx for item_idx in item_ids if item_idx in item_to_idx_inv]
    return [item_to_idx_inv[item_idx] for item_idx in valid_item_ids[:top_n]]
    
recommendations = []
csr_interractions_matrix = interaction_matrix.tocsr()
for user_id in user_ids:
    user_idx = user_to_idx[user_id]
    recs = recommend_for_user(user_idx, model, csr_interractions_matrix)
    for rec in recs:
        recommendations.append({'user_id': user_id, 'items': rec})

submission_df = pd.DataFrame(recommendations)

In [ ]:
submission_df.shape

(14039620, 2)

In [26]:
submission_df.to_csv('submission.csv', index=False)